# Git `restore` vs `reset`

Both `git restore` and `git reset` are used to undo things in Git, but they operate on **different concepts**.

---

# Big Picture

Git has 3 important areas:

```text
┌──────────────┐
│   Commits    │  ← Git history (HEAD)
└──────────────┘
        ↓
┌──────────────┐
│ Staging Area │  ← index
└──────────────┘
        ↓
┌──────────────┐
│ Working Tree │  ← your files
└──────────────┘
```

---

# Main Difference

| Command | Main Purpose |
|---|---|
| `git restore` | Restore files |
| `git reset` | Move HEAD / branch pointer |

---

# When Do We Use `restore`?

Use `restore` when:

- you changed a file accidentally
- you want to discard edits
- you want to unstage files
- you want to restore a file from another commit

It mainly affects:

- Working directory
- Staging area

It usually does **NOT** move commits or branches.

---

# When Do We Use `reset`?

Use `reset` when:

- you want to move HEAD backward
- remove commits
- rewrite local history
- uncommit changes
- go back to older commit states

It mainly affects:

- HEAD
- current branch
- optionally staging/worktree

---

# Understanding `HEAD`

`HEAD` means:

> "Where you currently are in history."

Example:

```text
A --- B --- C --- D   main
                  ↑
                 HEAD
```

You are currently at commit `D`.

---

# `git restore`

---

# 1. Discard Working Directory Changes

```bash
git restore file.txt
```

Before:

```text
Working Tree:
file.txt modified
```

After:

```text
file.txt restored to staged version
```

Equivalent old command:

```bash
git checkout -- file.txt
```

---

# 2. Unstage a File

```bash
git restore --staged file.txt
```

This removes file from staging area.

Before:

```text
Modified → Staged
```

After:

```text
Modified only
```

---

# 3. Restore From Specific Commit

```bash
git restore --source=COMMIT file.txt
```

Example:

```bash
git restore --source=HEAD~2 app.py
```

This restores file version from older commit.

---

# Summary of `restore`

| Command | Effect |
|---|---|
| `git restore file` | discard unstaged edits |
| `git restore --staged file` | unstage file |
| `git restore --source=commit file` | recover old version |

---

# `git reset`

This is more powerful and more dangerous.

---

# Core Idea of `reset`

It moves the branch pointer.

Example:

Before:

```text
A --- B --- C --- D   main
                  ↑
                 HEAD
```

Command:

```bash
git reset B
```

After:

```text
A --- B   main
      ↑
     HEAD
```

Commits `C` and `D` are no longer on branch history.

BUT:

They are usually still recoverable via:

```bash
git reflog
```

for some time.

---

# Important Modes of `reset`

---

# 1. `--soft`

```bash
git reset --soft B
```

Moves HEAD only.

---

## Before

```text
A --- B --- C --- D   main
                  ↑
                 HEAD
```

---

## After

```text
A --- B   main
      ↑
     HEAD
```

BUT:

All changes from `C` and `D` remain staged.

---

## Result

| Area | State |
|---|---|
| Commits | moved back |
| Staging | preserved |
| Files | preserved |

---

## Use Case

When:

- you want to rewrite commits
- squash commits
- recommit differently

---

# 2. `--mixed` (default)

```bash
git reset B
```

or

```bash
git reset --mixed B
```

Moves HEAD and unstages changes.

---

## Result

| Area | State |
|---|---|
| Commits | moved back |
| Staging | cleared |
| Files | preserved |

Changes become unstaged modifications.

---

# 3. `--hard`

```bash
git reset --hard B
```

Dangerous.

Moves HEAD AND deletes changes.

---

## Result

| Area | State |
|---|---|
| Commits | moved back |
| Staging | cleared |
| Files | overwritten |

Everything after commit `B` disappears from working tree.

---

# Visual Comparison

## `--soft`

```text
Commits: changed
Stage:   kept
Files:   kept
```

---

## `--mixed`

```text
Commits: changed
Stage:   cleared
Files:   kept
```

---

## `--hard`

```text
Commits: changed
Stage:   cleared
Files:   deleted/reset
```

---

# Important Question:
# What Happens To "Ahead" Commits?

Suppose:

```text
A --- B --- C --- D   main
                  ↑
                 HEAD
```

Then:

```bash
git reset --soft B
```

Now:

```text
A --- B   main
      ↑
     HEAD
```

Commits `C` and `D` are no longer reachable from `main`.

BUT:

Their changes still exist in staging area.

---

# If You Make New Commit Now

Suppose:

```bash
git commit -m "NEW"
```

History becomes:

```text
A --- B --- NEW   main
```

Now old commits `C` and `D` become orphaned/unreachable.

---

# Are We Creating A New Branch?

Technically:

- NO automatic branch is created.
- You are still on same branch (`main`).

But history diverges conceptually.

Old commits become detached from branch history.

---

# Can Old Commits Be Recovered?

Usually yes:

```bash
git reflog
```

Git keeps references temporarily.

Example:

```bash
git reflog
```

may show:

```text
D HEAD@{1}
C HEAD@{2}
```

Then:

```bash
git checkout D
```

or:

```bash
git branch old-history D
```

---

# Detached HEAD vs Reset

---

# Detached HEAD

Example:

```bash
git checkout B
```

or:

```bash
git switch --detach B
```

Now:

```text
A --- B --- C --- D
      ↑
     HEAD
```

HEAD points directly to commit, not branch.

If you commit now:

```text
A --- B --- X
 \
  C --- D main
```

You are outside branch history.

---

# Reset Is Different

```bash
git reset B
```

moves the branch itself.

```text
A --- B   main
```

No detached HEAD.

---

# Important Analogy

Think of commits as train stations.

```text
A → B → C → D
```

- `restore`:
  > Fixes files inside current station

- `reset`:
  > Moves the train backward to older station

---

# Common Practical Examples

---

# Undo Last Commit But Keep Changes

```bash
git reset --soft HEAD~1
```

---

# Undo Last Commit And Unstage Changes

```bash
git reset HEAD~1
```

---

# Completely Delete Last Commit

```bash
git reset --hard HEAD~1
```

---

# Discard File Changes

```bash
git restore file.txt
```

---

# Unstage File

```bash
git restore --staged file.txt
```

---

# Q & A

---

## Q1. Does `git restore` change history?

No.

It only changes files/staging.

---

## Q2. Does `git reset` change history?

Yes.

It moves branch pointer.

---

## Q3. Is `git reset` dangerous?

`--hard` can permanently remove work.

---

## Q4. Are removed commits immediately deleted?

Usually no.

Git keeps them temporarily in reflog.

---

## Q5. Does `reset` create a new branch?

No.

It moves current branch backward.

---

## Q6. If I commit after reset, what happens?

You create a new history path.

Old commits become unreachable.

---

## Q7. What is the safest reset?

```bash
git reset --soft
```

because files are preserved.

---

## Q8. How do I recover after bad reset?

Use:

```bash
git reflog
```

then:

```bash
git reset --hard COMMIT
```

---

# Final Mental Model

| Command | Think Of It As |
|---|---|
| `restore` | restore files |
| `reset` | move history backward |

---

# Recommended Beginner Rule

Use:

- `restore` → for files
- `reset --soft` → for undoing commits safely
- avoid `--hard` unless fully sure